# E13: OOT Evaluation and Model Selection

**Spec sections**: 5.4 (Q1/Q2 split reporting), 6.7 (L6), 8.1–8.5 (decision rules, metrics, paired bootstrap, poor results), 9.1 (Platt on DEV OOF), 9.3 (score mapping), D.3.  
**Inputs**: locked `model_registry.json` (all hashes re-verified), OOT GOOD+BAD via `common.load_window`, DEV (frozen E3 population), `oof_predictions.parquet`, L1 binning objects, `config/signoff.yaml`.  
**Outputs**: `calibration/platt_dev_oof.json`, `reports/05_oot.xlsx`, feature-log columns `oot_iv`, `oot_psi`, `oot_direction_ok`, `model_registry.json` key `oot_evaluation` (locked fields untouched), run summary, MLflow run.

Rule thresholds are read from the **registry** (`pre_registered`), never from config. Nothing is refitted on OOT and no feature is dropped or changed.

In [0]:
%pip install optbinning "numpy<2" "pyarrow==14.0.1" "statsmodels>=0.14.4" lightgbm interpret-core "scipy>=1.12,<1.14" openpyxl --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, json, time, pickle, shutil, tempfile, hashlib, importlib, copy, warnings
from datetime import datetime
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd
import yaml
import lightgbm as lgb
import interpret, optbinning
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", message="Missing values detected")

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)

ARTIFACTS  = common.artifacts_path(cfg)
REPORTS    = os.path.join(ARTIFACTS, "reports")
RUN_SUMS   = os.path.join(REPORTS, "run_summaries")
CALIB_DIR  = os.path.join(ARTIFACTS, "calibration")
REGISTRY_PATH = os.path.join(ARTIFACTS, "model_registry.json")
PLATT_PATH = os.path.join(CALIB_DIR, "platt_dev_oof.json")
XLSX_PATH  = os.path.join(REPORTS, "05_oot.xlsx")
for d in (REPORTS, RUN_SUMS, CALIB_DIR):
    os.makedirs(d, exist_ok=True)

cat, sch = cfg["databricks"]["catalog"], cfg["databricks"]["schema"]
MASTER_TBL = f"{cat}.{sch}.acard_master"
FLOG_TBL   = f"{cat}.{sch}.acard_feature_log"
id_col, date_col = cfg["columns"]["id"], cfg["columns"]["date"]
status_col = cfg["columns"]["status"]
LABELLED = cfg["labelled_statuses"]
PSI_STATUSES = cfg["psi_statuses"]
EPS = cfg["funnel"]["L3"]["eps"]
SPLITS = cfg["oot_report_splits"]                       # Q1 / Q2
PERIODS = ["overall"] + list(SPLITS)
MODELS = ["A", "B", "C"]
PAIRS = [("B", "A"), ("C", "A"), ("C", "B")]
PROPOSED_CUTOFF = cfg.get("strategy", {}).get("proposed_cutoff")
SEG_MIN_BADS = 50                                        # spec 8.2: segments with < 50 Bads -> monotonicity only
DIAG_GINI_DROP = 0.10                                    # spec 8.5: OOT Gini more than 0.10 below DEV OOF

GIT_COMMIT = common.git_commit(PROJECT_ROOT)
CONFIG_HASH = common.config_hash(cfg)

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(p):
    with open(p) as fh:
        return json.load(fh)

def _json_default(o):
    if hasattr(o, "item"):
        return o.item()
    if isinstance(o, float) and np.isnan(o):
        return None
    return str(o)

def write_json(obj, dst):
    """Local temp file then copy to the UC volume (FUSE does not allow chmod/utime)."""
    tmp = os.path.join(tempfile.mkdtemp(prefix="e13_"), os.path.basename(dst))
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2, default=_json_default)
    shutil.copyfile(tmp, dst)

def canonical(obj):
    return json.loads(json.dumps(obj, sort_keys=True, default=_json_default))

_t0 = time.time()
print(f"Project root: {PROJECT_ROOT}")
print(f"Git commit  : {GIT_COMMIT[:12]} | Config hash: {CONFIG_HASH[:12]}")
print(f"Periods     : {PERIODS} {SPLITS} | proposed_cutoff={PROPOSED_CUTOFF}")

In [0]:
REG0 = read_json(REGISTRY_PATH)
assert REG0.get("locked") is True, "model_registry.json is not locked: run E12 (s04_lock) and sign off first"
for k in ("decision_rules_signed_by", "signed_date"):
    assert str(REG0.get("signoff", {}).get(k, "")).strip(), f"registry sign-off field {k} is empty"

HASH_CHECK = {k: sha256_file(v["path"]) == v["sha256"] for k, v in REG0["files"].items()}
bad = [k for k, ok in HASH_CHECK.items() if not ok]
assert not bad, f"ABORT: {len(bad)} locked file(s) changed since the lock: {bad}"

# Snapshot of everything locked (all keys except the oot_evaluation added by this module)
LOCKED_SNAPSHOT = canonical({k: v for k, v in REG0.items() if k != "oot_evaluation"})
LOCKED_SHA = hashlib.sha256(json.dumps(LOCKED_SNAPSHOT, sort_keys=True).encode()).hexdigest()

# Thresholds and lists come from the REGISTRY, never from config
PRE = REG0["pre_registered"]
THR = {"r1_gini_lift_lower": PRE["decision_rules"]["R1_discrimination"]["threshold"],
       "c_over_b_gini_lift_lower": PRE["selection_config"]["c_over_b_gini_lift_lower"],
       "r3_tfold_std_ratio": PRE["decision_rules"]["R3_time_stability"]["tfold_std_ratio"],
       "r3_q2_ks_drop_margin": PRE["decision_rules"]["R3_time_stability"]["q2_ks_drop_margin"]}
BOOT = PRE["bootstrap"]
cfg_boot = copy.deepcopy(cfg)
cfg_boot["selection"]["bootstrap"] = BOOT                # common.* bootstrap helpers read the registry settings
assert PRE["calibration"]["method"] == "platt" and PRE["calibration"]["fit_on"] == "dev_oof"
A_FEATS = REG0["features"]["A"]
POOL = REG0["features"]["BC_pool"]
BC_COLS = REG0["features"]["BC_columns"]
T_FOLD = REG0["t_fold"]
EBM_CHECK = REG0["ebm_shape_check"]["result"]
FILES = {k: v["path"] for k, v in REG0["files"].items()}

# Feature-log snapshot (every column except the three L6 columns this module writes)
L6_COLS = ["oot_iv", "oot_psi", "oot_direction_ok"]
FLOG0 = spark.table(FLOG_TBL).toPandas()
FLOG0_SEL = FLOG0.drop(columns=[c for c in L6_COLS if c in FLOG0.columns]).astype(str).sort_values("feature").reset_index(drop=True)

print(f"Registry v{REG0['version']} locked at {REG0['locked_at']} by {REG0['signoff']['decision_rules_signed_by']}; "
      f"{len(HASH_CHECK)} file hashes re-verified")
print(f"Thresholds (registry): {THR} | bootstrap {BOOT}")
print(f"A {len(A_FEATS)} features | B/C pool {len(POOL)} | EBM shape check {EBM_CHECK} | feature log {FLOG0.shape}")

In [0]:
# Model lock is in place, so OOT may now be read (only through common.load_window)
master = spark.table(MASTER_TBL).filter("window IN ('DEV', 'OOT')").toPandas()
dev_all = common.load_window(master, "DEV", cfg)
oot_all = common.load_window(master, "OOT", cfg)
del master

# DEV GOOD+BAD = the frozen E3 population (as in E10/E11)
frozen_ids = set(pd.read_parquet(FILES["folds/r_folds.parquet"], columns=["app_id"])["app_id"].astype(str))
dev_gb = dev_all[dev_all[status_col].isin(LABELLED) & dev_all[id_col].astype(str).isin(frozen_ids)].reset_index(drop=True)
oot_gb = oot_all[oot_all[status_col].isin(LABELLED)].reset_index(drop=True)
common.assert_labelled(dev_gb, cfg); common.assert_labelled(oot_gb, cfg)
dev_p1 = dev_all[dev_all[status_col].isin(PSI_STATUSES)].reset_index(drop=True)     # PSI population (spec 6.4.1)
oot_p1 = oot_all[oot_all[status_col].isin(PSI_STATUSES)].reset_index(drop=True)

y_dev = (dev_gb[status_col] == "BAD").astype(int).to_numpy()
y_oot = (oot_gb[status_col] == "BAD").astype(int).to_numpy()
for name, n, b in (("DEV", len(dev_gb), y_dev.sum()), ("OOT", len(oot_gb), y_oot.sum())):
    exp = cfg["expected_counts"][name]
    assert n == exp["records"] and b == exp["bads"], f"{name} GOOD+BAD {n}/{b} != expected {exp}"

def period_of(df):
    d = pd.to_datetime(df[date_col])
    out = pd.Series(pd.NA, index=df.index, dtype=object)
    for q, (s, e) in SPLITS.items():
        out[(d >= s) & (d < e)] = q
    return out

oot_gb["period"] = period_of(oot_gb).to_numpy()
oot_p1["period"] = period_of(oot_p1).to_numpy()
assert oot_gb["period"].notna().all() and oot_p1["period"].notna().all(), "OOT record outside Q1/Q2"

def pmask(df, period):
    return np.ones(len(df), bool) if period == "overall" else (df["period"] == period).to_numpy()

COUNTS = pd.DataFrame([{"sample": "DEV GOOD+BAD", "period": "all", "records": len(dev_gb), "bads": int(y_dev.sum())},
                       {"sample": "DEV P1 (PSI)", "period": "all", "records": len(dev_p1), "bads": None}]
                      + [{"sample": "OOT GOOD+BAD", "period": p, "records": int(pmask(oot_gb, p).sum()),
                          "bads": int(y_oot[pmask(oot_gb, p)].sum())} for p in PERIODS]
                      + [{"sample": "OOT P1 (PSI)", "period": p, "records": int(pmask(oot_p1, p).sum()), "bads": None}
                         for p in PERIODS])
display(COUNTS)

In [0]:
fdict = pd.read_csv(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv"))
FDICT_SHA = sha256_file(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv"))
SPEC = read_json(FILES["models/model_spec.json"])
WOE_KW = SPEC["A"]["woe_transform"]
A_BP = pickle.load(open(FILES["models/A/binning_process.pkl"], "rb"))
A_LR = pickle.load(open(FILES["models/A/lr_sklearn.pkl"], "rb"))
B_EBM = pickle.load(open(FILES["models/B/ebm.pkl"], "rb"))
C_BST = lgb.Booster(model_file=FILES["models/C/lgbm.txt"])
assert C_BST.num_trees() == REG0["lightgbm_rounds"]["final_rounds"], "LightGBM trees != locked final rounds"
assert all(len(t) == 1 for t in B_EBM.term_features_), "EBM has interaction terms"

def score_raw(df):
    """Raw (uncalibrated) Bad probability of the locked A, B, C. Higher = riskier."""
    XA = df[A_FEATS].apply(pd.to_numeric, errors="coerce")
    pa = A_LR.predict_proba(np.asarray(A_BP.transform(XA, metric="woe", **WOE_KW), dtype=float))[:, 1]
    XR = common.prepare_raw(df, POOL, fdict)
    assert list(XR.columns) == BC_COLS, "prepare_raw columns differ from the locked B/C columns"
    pb = B_EBM.predict_proba(XR)[:, 1]
    pc = C_BST.predict(XR)
    return {"A": pa, "B": pb, "C": pc}

RAW = {"dev_gb": score_raw(dev_gb), "oot_gb": score_raw(oot_gb), "dev_p1": score_raw(dev_p1), "oot_p1": score_raw(oot_p1)}
for k, v in RAW.items():
    for m in MODELS:
        assert np.isfinite(v[m]).all() and ((v[m] > 0) & (v[m] < 1)).all(), f"{k} {m}: PD outside (0, 1)"
print({m: round(roc_auc_score(y_oot, RAW['oot_gb'][m]), 4) for m in MODELS}, "<- OOT raw AUC (point)")

In [0]:
oof = pd.read_parquet(FILES["oof_predictions.parquet"])
oof["app_id"] = oof["app_id"].astype(str)
ylab = pd.Series(y_dev, index=dev_gb[id_col].astype(str))
PLATT, OOF_P = {}, {}
for m in MODELS:
    g = oof[oof["model"] == m].set_index("app_id")
    assert len(g) == len(dev_gb) and set(g.index) == set(ylab.index), f"OOF for {m} does not cover DEV GOOD+BAD"
    yy = ylab.loc[g.index].to_numpy()
    PLATT[m] = common.fit_platt(yy, g["p_oof"].to_numpy())
    OOF_P[m] = (yy, g["p_oof"].to_numpy())

PLATT_DOC = {"module": "E13_oot", "run_time": datetime.now().isoformat(timespec="seconds"), "git_commit": GIT_COMMIT,
             "config_sha256": CONFIG_HASH, "registry_version": REG0["version"], "method": "platt",
             "formula": "logit(PD) = a + b * logit(p_raw)", "fit_on": "dev_oof",
             "oof_file": FILES["oof_predictions.parquet"], "oof_sha256": REG0["files"]["oof_predictions.parquet"]["sha256"],
             "logit_clip": 1e-6, "estimator": "common.fit_platt (unpenalised LogisticRegression, tol=1e-10)",
             "params": PLATT}
write_json(PLATT_DOC, PLATT_PATH)

CAL = {k: {m: common.apply_platt(v[m], PLATT[m]) for m in MODELS} for k, v in RAW.items()}
SCORE = {k: {m: common.pd_to_score(v[m], cfg) for m in MODELS} for k, v in CAL.items()}
# Calibration on DEV OOF must reproduce the DEV bad rate (MLE property of the intercept)
for m in MODELS:
    yy, pp = OOF_P[m]
    assert abs(common.apply_platt(pp, PLATT[m]).mean() - yy.mean()) < 1e-4, f"Platt {m}: mean PD != DEV bad rate"
display(pd.DataFrame(PLATT).T)
print(f"Saved {PLATT_PATH}")

In [0]:
def capture(y, p, share):
    k = int(np.ceil(share * len(p)))
    top = np.argsort(-p, kind="mergesort")[:k]
    return float(y[top].sum() / y.sum())

# DEV cut-points: deciles / quintiles of each model's calibrated DEV PD (GOOD+BAD, locked full-DEV model)
def edges(p, q):
    return np.unique(np.quantile(p, np.linspace(0, 1, q + 1)[1:-1]))
EDGES = {m: {"decile": edges(CAL["dev_gb"][m], 10), "quintile": edges(CAL["dev_gb"][m], 5)} for m in MODELS}
def band(p, e):
    return np.searchsorted(e, p, side="right") + 1          # 1 = lowest PD (best) ... 10 = highest PD

DEV_OOF = {}
for m in MODELS:
    yy, pp = OOF_P[m]
    a = roc_auc_score(yy, pp)
    DEV_OOF[m] = {"auc": a, "gini": 2 * a - 1, "ks": common.ks_stat(yy, pp)}

rows, cal_rows, psi_rows = [], [], []
for m in MODELS:
    for p in PERIODS:
        mk = pmask(oot_gb, p)
        yy, pr, pc = y_oot[mk], RAW["oot_gb"][m][mk], CAL["oot_gb"][m][mk]
        r = {"model": m, "period": p, **common.bootstrap_auc_gini_ks(yy, pr, cfg_boot),
             "top10_capture": capture(yy, pr, 0.10), "top20_capture": capture(yy, pr, 0.20)}
        r.update({"dev_oof_auc": DEV_OOF[m]["auc"], "dev_oof_gini": DEV_OOF[m]["gini"], "dev_oof_ks": DEV_OOF[m]["ks"],
                  "gini_decay": DEV_OOF[m]["gini"] - r["gini"], "ks_decay": DEV_OOF[m]["ks"] - r["ks"]})
        rows.append(r)
        cal_rows.append({"model": m, "period": p, **common.calibration_stats(yy, pc),
                         "log_loss_raw": common.calibration_stats(yy, pr)["log_loss"]})
        mk1 = pmask(oot_p1, p)
        e = EDGES[m]["decile"]
        psi_rows.append({"model": m, "period": p, "n_dev": len(dev_p1), "n_oot": int(mk1.sum()),
                         "score_psi": common.psi_from_bins(pd.Series(band(CAL["dev_p1"][m], e)),
                                                           pd.Series(band(CAL["oot_p1"][m][mk1], e)), EPS),
                         "score_psi_good_bad": common.psi_from_bins(pd.Series(band(CAL["dev_gb"][m], e)),
                                                                    pd.Series(band(CAL["oot_gb"][m][mk], e)), EPS)})
METRICS = pd.DataFrame(rows)
CALIB = pd.DataFrame(cal_rows)
SCORE_PSI = pd.DataFrame(psi_rows)
display(METRICS[["model", "period", "n", "n_bad", "auc", "auc_lo", "auc_hi", "gini", "gini_lo", "gini_hi", "ks", "ks_lo",
                 "ks_hi", "top10_capture", "top20_capture", "dev_oof_gini", "gini_decay"]].round(4))
display(CALIB.round(4)); display(SCORE_PSI.round(4))

In [0]:
band_rows, mono_rows = [], []
for m in MODELS:
    for kind in ("decile", "quintile"):
        e = EDGES[m][kind]
        dev_b = band(CAL["dev_gb"][m], e)
        for p in PERIODS:
            mk = pmask(oot_gb, p)
            b = band(CAL["oot_gb"][m][mk], e)
            t = pd.DataFrame({"band": b, "y": y_oot[mk], "pd": CAL["oot_gb"][m][mk], "score": SCORE["oot_gb"][m][mk]})
            g = t.groupby("band").agg(n=("y", "size"), bads=("y", "sum"), mean_pd=("pd", "mean"),
                                      score_min=("score", "min"), score_max=("score", "max"))
            g = g.reindex(range(1, len(e) + 2), fill_value=0)
            g["bad_rate"] = np.where(g["n"] > 0, g["bads"] / g["n"].replace(0, np.nan), np.nan)
            g["share"] = g["n"] / g["n"].sum()
            g["dev_share"] = pd.Series(dev_b).value_counts(normalize=True).reindex(g.index, fill_value=0).to_numpy()
            g["pd_upper_edge"] = list(e) + [1.0]
            g = g.reset_index().assign(model=m, kind=kind, period=p)
            band_rows.append(g)
            br = g["bad_rate"].dropna().to_numpy()
            mono_rows.append({"model": m, "kind": kind, "period": p, "n_bands": len(br),
                              "monotonic": bool(np.all(np.diff(br) >= 0)),
                              "n_reversals": int((np.diff(br) < 0).sum())})
BANDS = pd.concat(band_rows, ignore_index=True)
MONO = pd.DataFrame(mono_rows)
CAL_DECILE = BANDS[BANDS["kind"] == "decile"][["model", "period", "band", "n", "bads", "bad_rate", "mean_pd"]].copy()
CAL_DECILE["pd_minus_bad_rate"] = CAL_DECILE["mean_pd"] - CAL_DECILE["bad_rate"]
display(MONO.pivot_table(index=["model", "kind"], columns="period", values="monotonic"))

In [0]:
# Segment columns from config; watch columns = config watch minus feature_override (age is a feature; reported as an age band)
SEG_COLS = [c for c in cfg["columns"]["segments"] if c in oot_gb.columns]
SEG_MISSING = [c for c in cfg["columns"]["segments"] if c not in oot_gb.columns]
override = set(cfg["column_roles"].get("feature_override", []))
WATCH_COLS = [c for c in cfg["column_roles"]["watch"] if c not in override and c in oot_gb.columns]
AGE_COL = next((c for c in cfg["column_roles"]["watch"] if c in override and c in oot_gb.columns), None)
NOTES = []
if SEG_MISSING:
    NOTES.append(f"config.columns.segments not in master table (not reported): {SEG_MISSING}")
if AGE_COL:
    # The spec does not define age-band edges: fixed DEV quintile edges of AGE are used (report only)
    age_dev = pd.to_numeric(dev_gb[AGE_COL], errors="coerce")
    AGE_EDGES = np.unique(np.quantile(age_dev.dropna(), [0.2, 0.4, 0.6, 0.8]))
    def age_band(s):
        v = pd.to_numeric(s, errors="coerce")
        lab = np.searchsorted(AGE_EDGES, v, side="right")
        names = [f"<{AGE_EDGES[0]:g}"] + [f"[{a:g},{b:g})" for a, b in zip(AGE_EDGES[:-1], AGE_EDGES[1:])] + [f">={AGE_EDGES[-1]:g}"]
        return pd.Series(np.where(v.isna(), "MISSING", np.array(names, dtype=object)[np.minimum(lab, len(names) - 1)]), index=s.index)
    oot_gb["age_band"] = age_band(oot_gb[AGE_COL]).to_numpy()
    NOTES.append(f"age_band = fixed DEV quintile edges of {AGE_COL} {list(AGE_EDGES)} (spec gives no band edges; report only)")
GROUP_COLS = SEG_COLS + WATCH_COLS + (["age_band"] if AGE_COL else [])

seg_rows = []
for m in MODELS:
    e5 = EDGES[m]["quintile"]
    for p in PERIODS:
        mk = pmask(oot_gb, p)
        sub = oot_gb.loc[mk, GROUP_COLS].copy()
        sub["y"], sub["pd"] = y_oot[mk], RAW["oot_gb"][m][mk]
        sub["q5"] = band(CAL["oot_gb"][m][mk], e5)
        for gc in GROUP_COLS:
            for val, g in sub.groupby(sub[gc].astype(str).fillna("MISSING")):
                nb = int(g["y"].sum())
                br = g.groupby("q5")["y"].mean().reindex(range(1, len(e5) + 2))
                brv = br.dropna().to_numpy()
                r = {"model": m, "period": p, "variable": gc, "value": val,
                     "type": "watch" if gc in WATCH_COLS + ["age_band"] else "segment",
                     "n": len(g), "bads": nb, "bad_rate": g["y"].mean(),
                     "auc": roc_auc_score(g["y"], g["pd"]) if nb >= SEG_MIN_BADS and nb < len(g) else np.nan,
                     "report": "AUC + quintiles" if nb >= SEG_MIN_BADS else f"< {SEG_MIN_BADS} Bads: monotonicity only",
                     "quintile_monotonic": bool(np.all(np.diff(brv) >= 0)) if len(brv) > 1 else None}
                r.update({f"q{i}_bad_rate": br.loc[i] for i in br.index})
                seg_rows.append(r)
SEGMENTS = pd.DataFrame(seg_rows)

# Approval by watch variable: no proposed cut-off -> share of each group in each DEV score decile
appr_rows = []
for m in MODELS:
    e10 = EDGES[m]["decile"]
    for p in PERIODS:
        mk = pmask(oot_gb, p)
        dec = band(CAL["oot_gb"][m][mk], e10)
        sc = SCORE["oot_gb"][m][mk]
        for gc in WATCH_COLS + (["age_band"] if AGE_COL else []):
            vals = oot_gb.loc[mk, gc].astype(str).fillna("MISSING").to_numpy()
            for val in np.unique(vals):
                sel = vals == val
                r = {"model": m, "period": p, "variable": gc, "value": val, "n": int(sel.sum())}
                if PROPOSED_CUTOFF is not None:
                    r["approval_rate_at_cutoff"] = float((sc[sel] >= PROPOSED_CUTOFF).mean())
                r.update({f"decile{d}_share": float((dec[sel] == d).mean()) for d in range(1, len(e10) + 2)})
                appr_rows.append(r)
APPROVAL = pd.DataFrame(appr_rows)
if PROPOSED_CUTOFF is None:
    NOTES.append("strategy.proposed_cutoff is null: approval rate at cut-off not computed; watch-group shares by DEV "
                 "score decile reported instead (decile 1 = lowest PD).")
print(f"Segments: {SEG_COLS} | watch: {WATCH_COLS} | age: {AGE_COL}")
for n in NOTES:
    print("NOTE:", n)
display(SEGMENTS[(SEGMENTS.period == "overall")].head(20))

In [0]:
pb_rows, dl_rows = [], []
for x, z in PAIRS:
    for p in PERIODS:
        mk = pmask(oot_gb, p)
        res = common.paired_bootstrap(y_oot[mk], RAW["oot_gb"][x][mk], RAW["oot_gb"][z][mk], cfg_boot)
        pb_rows.append(res.assign(pair=f"{x}-{z}", period=p))
        dl = common.delong_test(y_oot[mk], RAW["oot_gb"][x][mk], RAW["oot_gb"][z][mk])
        bsa = res.set_index("metric").loc["dAUC"]
        dl_rows.append({"pair": f"{x}-{z}", "period": p, **dl, "boot_dAUC_lo": bsa["ci_low"], "boot_dAUC_hi": bsa["ci_high"],
                        "agree_sign_of_lower_bound": bool((dl["ci_low"] > 0) == (bsa["ci_low"] > 0))})
PAIRED = pd.concat(pb_rows, ignore_index=True)[["pair", "period", "metric", "mean", "ci_low", "ci_high", "p_le_0"]]
DELONG = pd.DataFrame(dl_rows)
display(PAIRED[PAIRED.period == "overall"].round(4))
display(DELONG.round(4))

In [0]:
with open(os.path.join(PROJECT_ROOT, cfg["review_files"]["signoff"])) as fh:
    SIGNOFF = yaml.safe_load(fh) or {}
R4_APPROVED = SIGNOFF.get("r4_reason_codes_approved") is True

met = METRICS.set_index(["model", "period"])
cal = CALIB.set_index(["model", "period"])
pb = PAIRED.set_index(["pair", "period", "metric"])
ks_drop = {m: float(met.loc[(m, "Q1"), "ks"] - met.loc[(m, "Q2"), "ks"]) for m in MODELS}

RULES = {}
for x in ("B", "C"):
    lo = float(pb.loc[(f"{x}-A", "overall", "dGini"), "ci_low"])
    r1 = lo > THR["r1_gini_lift_lower"]
    llx, lla = float(cal.loc[(x, "overall"), "log_loss"]), float(cal.loc[("A", "overall"), "log_loss"])
    r2 = llx <= lla
    sx, sa = T_FOLD[x]["t_auc_std"], T_FOLD["A"]["t_auc_std"]
    r3a = sx <= THR["r3_tfold_std_ratio"] * sa
    r3b = ks_drop[x] <= ks_drop["A"] + THR["r3_q2_ks_drop_margin"]
    if x == "B" and EBM_CHECK != "PASS":
        r4, r4_why = "FAIL", f"EBM shape check = {EBM_CHECK} (spec 7.4); reason-code approval = {R4_APPROVED}"
    elif R4_APPROVED:
        r4, r4_why = "PASS", "signoff r4_reason_codes_approved = true" + ("; EBM shape check PASS" if x == "B" else "")
    else:
        r4, r4_why = "PENDING", "signoff r4_reason_codes_approved is not true (compliance approval of reason codes pending)"
    r123 = r1 and r2 and r3a and r3b
    qual = "YES" if (r123 and r4 == "PASS") else ("PENDING_R4" if (r123 and r4 == "PENDING") else "NO")
    RULES[x] = {
        "R1": {"pass": bool(r1), "value": lo, "threshold": THR["r1_gini_lift_lower"],
               "reason": f"lower 95% CI of dGini({x}-A) = {lo:.4f} {'>' if r1 else '<='} {THR['r1_gini_lift_lower']}"},
        "R2": {"pass": bool(r2), "value": llx, "benchmark": lla,
               "reason": f"calibrated OOT log loss {x} {llx:.5f} {'<=' if r2 else '>'} A {lla:.5f}"},
        "R3": {"pass": bool(r3a and r3b), "tfold_std": sx, "tfold_std_A": sa, "ratio_threshold": THR["r3_tfold_std_ratio"],
               "ks_drop": ks_drop[x], "ks_drop_A": ks_drop["A"], "margin": THR["r3_q2_ks_drop_margin"],
               "reason": (f"T-fold AUC std {sx:.4f} {'<=' if r3a else '>'} {THR['r3_tfold_std_ratio']} x {sa:.4f}; "
                          f"KS drop Q1->Q2 {ks_drop[x]:.4f} {'<=' if r3b else '>'} A {ks_drop['A']:.4f} + {THR['r3_q2_ks_drop_margin']}")},
        "R4": {"status": r4, "reason": r4_why},
        "qualifies": qual,
    }

def choose(qB, qC):
    """Spec 8.1 selection logic; qX = True if challenger X qualifies."""
    if not qB and not qC:
        return "A", "neither B nor C qualifies -> A"
    if qB != qC:
        w = "B" if qB else "C"
        return w, f"only {w} qualifies -> {w}"
    lo = float(pb.loc[("C-B", "overall", "dGini"), "ci_low"])
    if lo > THR["c_over_b_gini_lift_lower"]:
        return "C", f"both qualify; lower CI of dGini(C-B) = {lo:.4f} > {THR['c_over_b_gini_lift_lower']} -> C"
    return "B", f"both qualify; lower CI of dGini(C-B) = {lo:.4f} <= {THR['c_over_b_gini_lift_lower']} -> B (white-box preferred)"

ANY_PENDING = any(RULES[x]["R4"]["status"] == "PENDING" for x in RULES)
# Provisional choice: PENDING R4 treated as passing; fallback: PENDING R4 treated as failing
SELECTED, SEL_REASON = choose(RULES["B"]["qualifies"] != "NO", RULES["C"]["qualifies"] != "NO")
FALLBACK, FB_REASON = choose(RULES["B"]["qualifies"] == "YES", RULES["C"]["qualifies"] == "YES")
STATUS = "PROVISIONAL" if ANY_PENDING else "FINAL"

RULE_TABLE = pd.DataFrame([{"challenger": x, "rule": r, "result": (v["status"] if r == "R4" else ("PASS" if v["pass"] else "FAIL")),
                            "reason": v["reason"]} for x in RULES for r, v in RULES[x].items() if r != "qualifies"]
                          + [{"challenger": x, "rule": "qualifies", "result": RULES[x]["qualifies"], "reason": ""} for x in RULES])
display(RULE_TABLE)
print(f"\nSELECTED MODEL: {SELECTED} ({STATUS}) - {SEL_REASON}")
if ANY_PENDING:
    print(f"If the PENDING R4 is not approved: {FALLBACK} - {FB_REASON}")

In [0]:
# For every feature used by A, B or C: L1 (DEV-fitted) bins applied to OOT. Nothing is refitted, nothing is dropped.
L6_FEATS = list(dict.fromkeys(A_FEATS + POOL))
BINOBJ = {f: pickle.load(open(FILES[f"binning/L1/{f}.pkl"], "rb")) for f in L6_FEATS}
fdx = fdict.set_index("feature")

def bin_idx(f, s):
    ob = BINOBJ[f]
    x = s if str(getattr(ob, "dtype", "numerical")) == "categorical" else pd.to_numeric(s, errors="coerce")
    return np.asarray(ob.transform(np.asarray(x), metric="indices"))

def n_regular(f):
    ob = BINOBJ[f]
    return len(ob.splits) + (0 if str(getattr(ob, "dtype", "numerical")) == "categorical" else 1)

def bin_stats(idx, y):
    t = pd.DataFrame({"bin": idx, "y": y}).groupby("bin")["y"].agg(n="size", bads="sum")
    t["goods"] = t["n"] - t["bads"]
    pb_, pg_ = (t["bads"] / t["bads"].sum()).clip(lower=EPS), (t["goods"] / t["goods"].sum()).clip(lower=EPS)
    t["bad_rate"] = t["bads"] / t["n"]
    t["woe"] = np.log(pg_ / pb_)
    t["iv"] = (pg_ - pb_) * t["woe"]
    return t

l6_rows, l6_bins = [], []
for f in L6_FEATS:
    nreg = n_regular(f)
    d_idx = bin_idx(f, dev_gb[f])
    d_tab = bin_stats(d_idx, y_dev)
    d1_idx = pd.Series(bin_idx(f, dev_p1[f]))
    o_idx_all = bin_idx(f, oot_gb[f])
    o1_idx_all = bin_idx(f, oot_p1[f])
    for p in PERIODS:
        mk, mk1 = pmask(oot_gb, p), pmask(oot_p1, p)
        o_tab = bin_stats(o_idx_all[mk], y_oot[mk])
        both = [b for b in d_tab.index if b < nreg and b in o_tab.index]
        if len(both) >= 3:
            rho = spearmanr(d_tab.loc[both, "bad_rate"], o_tab.loc[both, "bad_rate"]).correlation
            ok = bool(rho > 0)
        elif len(both) == 2:
            rho = np.nan
            ok = bool(np.sign(np.diff(d_tab.loc[both, "bad_rate"]))[0] == np.sign(np.diff(o_tab.loc[both, "bad_rate"]))[0])
        else:
            rho, ok = np.nan, None
        l6_rows.append({"feature": f, "period": p, "in_A": f in A_FEATS, "in_BC": f in POOL,
                        "dev_iv": float(d_tab["iv"].sum()), "oot_iv": float(o_tab["iv"].sum()),
                        "oot_psi": common.psi_from_bins(d1_idx, pd.Series(o1_idx_all[mk1]), EPS),
                        "n_regular_bins": nreg, "bins_compared": len(both), "bad_rate_rank_corr": rho,
                        "oot_direction_ok": ok})
        bt = d_tab.add_prefix("dev_").join(o_tab.add_prefix("oot_"), how="outer").reset_index()
        bt["bin_type"] = np.where(bt["bin"] < nreg, "regular", "special/missing")
        l6_bins.append(bt.assign(feature=f, period=p))
L6 = pd.DataFrame(l6_rows)
L6_BINS = pd.concat(l6_bins, ignore_index=True)

# Feature log: write only oot_iv, oot_psi, oot_direction_ok (overall); every other column untouched
l6o = L6[L6.period == "overall"].set_index("feature")
fl = FLOG0.drop(columns=[c for c in L6_COLS if c in FLOG0.columns]).copy()
fl["oot_iv"] = fl["feature"].map(l6o["oot_iv"])
fl["oot_psi"] = fl["feature"].map(l6o["oot_psi"])
fl["oot_direction_ok"] = fl["feature"].map(l6o["oot_direction_ok"].map({True: "Y", False: "N"}))
assert len(fl) == len(FLOG0) and fl["feature"].is_unique, "feature-log row count changed"
fl = fl[list(FLOG0.columns) + [c for c in L6_COLS if c not in FLOG0.columns]]
for c in fl.columns:
    if fl[c].dtype == object:
        fl[c] = fl[c].where(fl[c].notna(), None)
    if fl[c].isna().all():
        fl[c] = fl[c].astype(str)
spark.createDataFrame(fl).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FLOG_TBL)
for pth in (os.path.join(ARTIFACTS, "feature_log.csv"), os.path.join(REPORTS, "feature_log.csv")):
    fl.to_csv(pth, index=False)
print(f"L6: {len(L6_FEATS)} features x {len(PERIODS)} periods | direction not OK (overall): "
      f"{l6o.index[l6o['oot_direction_ok'] == False].tolist()} | OOT PSI > 0.25: {l6o.index[l6o['oot_psi'] > cfg['funnel']['L3']['psi_red']].tolist()}")
display(L6[L6.period == "overall"].round(4))

In [0]:
ov = METRICS[METRICS.period == "overall"].set_index("model")
DIAG_TRIGGERED = bool((ov["gini_decay"] > DIAG_GINI_DROP).all())
if DIAG_TRIGGERED:
    DIAGNOSIS = pd.concat([
        METRICS[["model", "period", "gini", "dev_oof_gini", "gini_decay", "ks", "bad_rate"]].assign(view="by_period"),
        SEGMENTS[SEGMENTS.period == "overall"][["model", "variable", "value", "n", "bads", "auc"]].assign(view="by_segment"),
        L6[L6.period == "overall"][["feature", "dev_iv", "oot_iv", "oot_psi", "oot_direction_ok"]].assign(view="by_feature"),
    ], ignore_index=True)
    NOTES.append(f"Spec 8.5: every model's OOT Gini is more than {DIAG_GINI_DROP} below DEV OOF; diagnosis sheet added. "
                 "No feature or parameter is changed; any change must be disclosed and agreed with validation.")
else:
    DIAGNOSIS = pd.DataFrame([{"triggered": False, "rule": f"all models gini_decay > {DIAG_GINI_DROP}",
                               **{f"{m}_gini_decay": float(ov.loc[m, 'gini_decay']) for m in MODELS}}])

SUMMARY = pd.DataFrame([
    {"item": "selected_model", "value": SELECTED}, {"item": "status", "value": STATUS},
    {"item": "selection_reason", "value": SEL_REASON},
    {"item": "fallback_if_pending_R4_fails", "value": f"{FALLBACK}: {FB_REASON}" if ANY_PENDING else "n/a"},
    {"item": "n_comparisons", "value": "2 challengers vs benchmark (B-A, C-A) + C-B if both qualify"},
    {"item": "thresholds_source", "value": f"model_registry.json v{REG0['version']} pre_registered"},
    *[{"item": f"threshold.{k}", "value": v} for k, v in THR.items()],
    {"item": "bootstrap", "value": json.dumps(BOOT)},
    {"item": "diagnosis_triggered", "value": DIAG_TRIGGERED},
    *[{"item": "note", "value": n} for n in NOTES],
])
meta = pd.DataFrame([{"module": "E13_oot", "run_time": datetime.now().isoformat(timespec="seconds"), "git_commit": GIT_COMMIT,
                      "config_sha256": CONFIG_HASH, "registry_version": REG0["version"], "registry_locked_sha256": LOCKED_SHA,
                      "feature_dict_sha256": FDICT_SHA}])
SHEETS = {"meta": meta, "summary": SUMMARY, "rules": RULE_TABLE, "counts": COUNTS, "metrics": METRICS,
          "calibration": CALIB, "platt": pd.DataFrame(PLATT).T.reset_index().rename(columns={"index": "model"}),
          "bands": BANDS, "band_monotonic": MONO, "calibration_by_decile": CAL_DECILE, "score_psi": SCORE_PSI,
          "segments": SEGMENTS, "approval_by_watch": APPROVAL, "paired_bootstrap": PAIRED, "delong": DELONG,
          "L6_summary": L6, "L6_bins": L6_BINS, "diagnosis": DIAGNOSIS}
tmp = os.path.join(tempfile.mkdtemp(prefix="e13_"), "05_oot.xlsx")
with pd.ExcelWriter(tmp, engine="openpyxl") as w:
    for name, df in SHEETS.items():
        df.to_excel(w, sheet_name=name[:31], index=False)
shutil.copyfile(tmp, XLSX_PATH)
print(f"Saved {XLSX_PATH} ({len(SHEETS)} sheets) | diagnosis triggered: {DIAG_TRIGGERED}")

In [0]:
REG1 = read_json(REGISTRY_PATH)
assert canonical({k: v for k, v in REG1.items() if k != "oot_evaluation"}) == LOCKED_SNAPSHOT, \
    "locked registry fields changed during E13 - abort"
r4 = lambda v: round(float(v), 6)
REG1["oot_evaluation"] = {
    "module": "E13_oot", "run_time": datetime.now().isoformat(timespec="seconds"), "git_commit": GIT_COMMIT,
    "config_sha256": CONFIG_HASH, "locked_content_sha256": LOCKED_SHA,
    "selected_model": SELECTED, "status": STATUS, "selection_reason": SEL_REASON,
    "fallback_if_pending_r4_fails": {"model": FALLBACK, "reason": FB_REASON} if ANY_PENDING else None,
    "thresholds_used": THR, "bootstrap_used": BOOT,
    "rules": canonical(RULES),
    "metrics": {m: {p: {k: r4(met.loc[(m, p), k]) for k in ("auc", "auc_lo", "auc_hi", "gini", "gini_lo", "gini_hi",
                                                             "ks", "ks_lo", "ks_hi", "gini_decay")}
                    | {k: r4(cal.loc[(m, p), k]) for k in ("citl", "cal_slope", "brier", "log_loss")}
                    for p in PERIODS} for m in MODELS},
    "paired_bootstrap_dgini": {f"{x}-{z}": {p: {k: r4(pb.loc[(f"{x}-{z}", p, "dGini"), k]) for k in ("mean", "ci_low", "ci_high", "p_le_0")}
                                            for p in PERIODS} for x, z in PAIRS},
    "calibration_file": {"path": PLATT_PATH, "sha256": sha256_file(PLATT_PATH)},
    "report": {"path": XLSX_PATH, "sha256": sha256_file(XLSX_PATH)},
    "diagnosis_triggered": DIAG_TRIGGERED, "notes": NOTES,
}
write_json(REG1, REGISTRY_PATH)
print(f"model_registry.json: oot_evaluation written (selected {SELECTED}, {STATUS}); locked fields unchanged")

In [0]:
import mlflow
mlflow.set_experiment(cfg["databricks"]["mlflow_experiment"])
with mlflow.start_run(run_name="E13_oot") as run:
    mlflow.set_tags({"module": "E13_oot", "git_commit": GIT_COMMIT, "selected_model": SELECTED, "status": STATUS,
                     "registry_version": REG0["version"]})
    mlflow.log_params({"config_hash": CONFIG_HASH[:12], **THR, "bootstrap_n": BOOT["n"], "bootstrap_seed": BOOT["seed"]})
    mlflow.log_metrics({f"{m}_{p}_{k}": float(met.loc[(m, p), k]) for m in MODELS for p in PERIODS for k in ("auc", "gini", "ks")})
    mlflow.log_metrics({f"{m}_{p}_logloss_cal": float(cal.loc[(m, p), "log_loss"]) for m in MODELS for p in PERIODS})
    for pth in (XLSX_PATH, PLATT_PATH, REGISTRY_PATH):
        mlflow.log_artifact(pth)
    MLFLOW_RUN_ID = run.info.run_id
print(f"MLflow run {MLFLOW_RUN_ID}")

In [0]:
# ── Acceptance (E13): re-read outputs from the volume / table and check every criterion ──
print("=" * 60); print("ACCEPTANCE CRITERIA (E13 — OOT evaluation and selection)"); print("=" * 60)
REG2 = read_json(REGISTRY_PATH)
ev = REG2["oot_evaluation"]
xl = pd.ExcelFile(XLSX_PATH)
flog_now = spark.table(FLOG_TBL).toPandas()
flog_now_sel = flog_now.drop(columns=[c for c in L6_COLS if c in flog_now.columns]).astype(str).sort_values("feature").reset_index(drop=True)
platt_disk = read_json(PLATT_PATH)

# Independent re-derivation of the choice from the registry record alone
def _q(x):
    rr = ev["rules"][x]
    return rr["R1"]["pass"] and rr["R2"]["pass"] and rr["R3"]["pass"] and rr["R4"]["status"] in ("PASS", "PENDING")
reder = choose(_q("B"), _q("C"))[0]
thr_reg = {"r1_gini_lift_lower": REG2["pre_registered"]["decision_rules"]["R1_discrimination"]["threshold"],
           "c_over_b_gini_lift_lower": REG2["pre_registered"]["selection_config"]["c_over_b_gini_lift_lower"],
           "r3_tfold_std_ratio": REG2["pre_registered"]["decision_rules"]["R3_time_stability"]["tfold_std_ratio"],
           "r3_q2_ks_drop_margin": REG2["pre_registered"]["decision_rules"]["R3_time_stability"]["q2_ks_drop_margin"]}
used_in_rules = {"r1_gini_lift_lower": {ev["rules"][x]["R1"]["threshold"] for x in "BC"},
                 "r3_tfold_std_ratio": {ev["rules"][x]["R3"]["ratio_threshold"] for x in "BC"},
                 "r3_q2_ks_drop_margin": {ev["rules"][x]["R3"]["margin"] for x in "BC"}}

def has_periods(df, keys):
    return all(set(g["period"]) == set(PERIODS) for _, g in df.groupby(keys))

# Independent AUC check: Mann-Whitney U rank formula vs roc_auc_score used in the metrics
from scipy.stats import rankdata
def auc_rank(y, p):
    rk = rankdata(p); n1 = y.sum(); n0 = len(y) - n1
    return (rk[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
auc_diff = max(abs(auc_rank(y_oot[pmask(oot_gb, p)], RAW["oot_gb"][m][pmask(oot_gb, p)]) - met.loc[(m, p), "auc"])
               for m in MODELS for p in PERIODS)

checks = [
    (f"Exactly one selected model ({ev['selected_model']}, {ev['status']}) with rule-by-rule reasons for B and C",
     ev["selected_model"] in MODELS and all(ev["rules"][x][r]["reason"] for x in "BC" for r in ("R1", "R2", "R3", "R4"))),
    (f"Choice re-derived from the recorded rule results = {reder}", reder == ev["selected_model"]),
    (f"PROVISIONAL label iff an R4 is PENDING (pending: {ANY_PENDING})", (ev["status"] == "PROVISIONAL") == ANY_PENDING),
    ("Thresholds used equal the registry pre_registered values",
     ev["thresholds_used"] == thr_reg and all(v == {thr_reg[k]} for k, v in used_in_rules.items())
     and ev["bootstrap_used"] == REG2["pre_registered"]["bootstrap"]),
    ("Every metric reported overall, Q1 and Q2 (metrics, calibration, bands, PSI, segments, bootstrap, DeLong, L6)",
     has_periods(METRICS, ["model"]) and has_periods(CALIB, ["model"]) and has_periods(BANDS, ["model", "kind"])
     and has_periods(SCORE_PSI, ["model"]) and has_periods(SEGMENTS, ["model", "variable"])
     and has_periods(PAIRED, ["pair", "metric"]) and has_periods(DELONG, ["pair"]) and has_periods(L6, ["feature"])),
    (f"AUC re-computed by rank formula matches (max diff {auc_diff:.1e})", auc_diff < 1e-9),
    ("Platt fitted on DEV OOF only; saved file equals the parameters used",
     platt_disk["fit_on"] == "dev_oof" and platt_disk["params"] == canonical(PLATT)),
    (f"Feature log: only {L6_COLS} written; all selection columns unchanged ({len(flog_now)} rows)",
     len(flog_now) == len(FLOG0) and flog_now_sel.equals(FLOG0_SEL) and all(c in flog_now.columns for c in L6_COLS)),
    (f"L6 covers every A/B/C feature ({len(L6_FEATS)})", set(L6["feature"]) == set(A_FEATS) | set(POOL)),
    ("Locked registry fields unchanged; locked = true",
     canonical({k: v for k, v in REG2.items() if k != "oot_evaluation"}) == LOCKED_SNAPSHOT and REG2["locked"] is True),
    ("Locked file hashes still verify", all(sha256_file(v["path"]) == v["sha256"] for v in REG2["files"].values())),
    (f"Report has meta sheet first and {len(SHEETS)} sheets", xl.sheet_names[0] == "meta" and len(xl.sheet_names) == len(SHEETS)),
]
PASS_S, FAIL_S = "\u2705 PASS", "\u274c FAIL"
all_pass = True
for desc, ok in checks:
    ok = bool(ok); all_pass &= ok
    print(f"  {PASS_S if ok else FAIL_S} : {desc}")
for n in NOTES:
    print(f"  NOTE : {n}")

summary = {"module": "E13_oot", "run_time": datetime.now().isoformat(timespec="seconds"), "elapsed_s": round(time.time() - _t0, 1),
           "git_commit": GIT_COMMIT, "config_hash": CONFIG_HASH, "registry_version": REG2["version"],
           "rows_dev_good_bad": len(dev_gb), "rows_oot_good_bad": len(oot_gb), "bads_oot": int(y_oot.sum()),
           "selected_model": SELECTED, "status": STATUS, "selection_reason": SEL_REASON,
           **{f"{m}_oot_auc": round(float(met.loc[(m, 'overall'), 'auc']), 4) for m in MODELS},
           **{f"{m}_oot_gini": round(float(met.loc[(m, 'overall'), 'gini']), 4) for m in MODELS},
           **{f"{m}_oot_ks": round(float(met.loc[(m, 'overall'), 'ks']), 4) for m in MODELS},
           "diagnosis_triggered": DIAG_TRIGGERED, "mlflow_run_id": MLFLOW_RUN_ID, "notes": NOTES,
           "checks": {d: bool(o) for d, o in checks}, "pass_fail": "PASS" if all_pass else "FAIL"}
write_json(summary, os.path.join(RUN_SUMS, "s05_oot.json"))
print(f"\nRun summary: {os.path.join(RUN_SUMS, 's05_oot.json')}")
for desc, ok in checks:
    assert ok, f"ACCEPTANCE FAILED: {desc}"
print("\n\u2705 All E13 acceptance criteria passed.")

In [0]:
# ── Predict STRESS period with models A, B, C and save results ──
# Uses the STRESS window data (2026-03 onward) for scoring only (no labels, per spec Ch.11).
# Saves predictions + key identifiers so the user can trace every case back to the master data.

# 1. Build the STRESS subset (master was deleted in the load cell; re-read and go through common.load_window)
stress_df = common.load_window(spark.table(MASTER_TBL).filter("window = 'STRESS'").toPandas(), "STRESS", cfg).reset_index(drop=True)
assert len(stress_df) > 0, "No STRESS rows in master"
print(f"STRESS records: {len(stress_df):,}")

# 2-3. Raw predictions with the locked models via score_raw (A: WoE + LR, B: EBM, C: LightGBM Booster)
raw_s = score_raw(stress_df)
for m in MODELS:
    stress_df[f"prob_{m}_raw"] = raw_s[m]

# 4. Calibrated PD (Platt fitted on DEV OOF: logit(PD) = a + b * logit(p_raw)) and score (spec 9.3)
for m in MODELS:
    stress_df[f"prob_{m}_cal"] = common.apply_platt(raw_s[m], PLATT[m])
    stress_df[f"score_{m}"] = common.pd_to_score(stress_df[f"prob_{m}_cal"].to_numpy(), cfg)

# 5. Keep key identifiers + watch / segment columns + predictions (no target: STRESS labels are not used, spec 11.1)
id_cols = [c for c in cfg["column_roles"]["key"] if c in stress_df.columns]
perf_cols = ["sample_status", "window", "app_month"]
watch_cols = [c for c in (SEG_COLS + WATCH_COLS) if c in stress_df.columns]
pred_cols = [c for c in stress_df.columns if c.startswith(("prob_", "score_"))]

keep_cols = list(dict.fromkeys(id_cols + perf_cols + watch_cols + pred_cols))
STRESS_PRED = stress_df[keep_cols].copy()
STRESS_PRED.attrs = {}   # Spark Connect toPandas() stores PlanMetrics in attrs; pyarrow cannot JSON-serialise them

# 6. Save to reports/ (local temp file then copy, as for other volume writes) and as a Delta table
_tmp = tempfile.mkdtemp(prefix="e13_stress_")
stress_csv = os.path.join(REPORTS, "stress_predictions.csv")
stress_parquet = os.path.join(REPORTS, "stress_predictions.parquet")
STRESS_PRED.to_csv(os.path.join(_tmp, "p.csv"), index=False)
shutil.copyfile(os.path.join(_tmp, "p.csv"), stress_csv)
STRESS_PRED.to_parquet(os.path.join(_tmp, "p.parquet"), index=False)
shutil.copyfile(os.path.join(_tmp, "p.parquet"), stress_parquet)

catalog = cfg["databricks"]["catalog"]
schema = cfg["databricks"]["schema"]
stress_tbl = f"{catalog}.{schema}.acard_stress_predictions"
spark.createDataFrame(STRESS_PRED).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(stress_tbl)

print(f"\nSaved {len(STRESS_PRED):,} STRESS predictions")
print(f"  CSV     : {stress_csv}")
print(f"  Parquet : {stress_parquet}")
print(f"  Table   : {stress_tbl}")
print(f"  Columns : {keep_cols}")
print(f"\nPrediction summary (calibrated):")
for m in MODELS:
    col = f"prob_{m}_cal"
    print(f"  Model {m}: mean={STRESS_PRED[col].mean():.4f}  std={STRESS_PRED[col].std():.4f}  "
          f"min={STRESS_PRED[col].min():.4f}  max={STRESS_PRED[col].max():.4f}")

display(STRESS_PRED.head(20))

In [0]:
# ── Predict FWD period with models A, B, C and save results ──
# Uses the FWD window data for scoring only (no labels, per spec Ch.11 analogy).
# Saves predictions + key identifiers so the user can trace every case back to the master data.

# 1. Build the FWD subset (re-read from master table)
fwd_df = common.load_window(spark.table(MASTER_TBL).filter("window = 'FWD'").toPandas(), "FWD", cfg).reset_index(drop=True)
assert len(fwd_df) > 0, "No FWD rows in master"
print(f"FWD records: {len(fwd_df):,}")

# 2-3. Raw predictions with the locked models via score_raw (A: WoE + LR, B: EBM, C: LightGBM Booster)
raw_f = score_raw(fwd_df)
for m in MODELS:
    fwd_df[f"prob_{m}_raw"] = raw_f[m]

# 4. Calibrated PD (Platt fitted on DEV OOF: logit(PD) = a + b * logit(p_raw)) and score (spec 9.3)
for m in MODELS:
    fwd_df[f"prob_{m}_cal"] = common.apply_platt(raw_f[m], PLATT[m])
    fwd_df[f"score_{m}"] = common.pd_to_score(fwd_df[f"prob_{m}_cal"].to_numpy(), cfg)

# 5. Keep key identifiers + watch / segment columns + predictions
id_cols = [c for c in cfg["column_roles"]["key"] if c in fwd_df.columns]
perf_cols = ["sample_status", "window", "app_month"]
watch_cols = [c for c in (SEG_COLS + WATCH_COLS) if c in fwd_df.columns]
pred_cols = [c for c in fwd_df.columns if c.startswith(("prob_", "score_"))]

keep_cols = list(dict.fromkeys(id_cols + perf_cols + watch_cols + pred_cols))
FWD_PRED = fwd_df[keep_cols].copy()
FWD_PRED.attrs = {}   # Spark Connect toPandas() stores PlanMetrics in attrs; pyarrow cannot JSON-serialise them

# 6. Save to reports/ (local temp file then copy, as for other volume writes) and as a Delta table
_tmp_fwd = tempfile.mkdtemp(prefix="e13_fwd_")
fwd_csv = os.path.join(REPORTS, "fwd_predictions.csv")
fwd_parquet = os.path.join(REPORTS, "fwd_predictions.parquet")
FWD_PRED.to_csv(os.path.join(_tmp_fwd, "p.csv"), index=False)
shutil.copyfile(os.path.join(_tmp_fwd, "p.csv"), fwd_csv)
FWD_PRED.to_parquet(os.path.join(_tmp_fwd, "p.parquet"), index=False)
shutil.copyfile(os.path.join(_tmp_fwd, "p.parquet"), fwd_parquet)

catalog = cfg["databricks"]["catalog"]
schema = cfg["databricks"]["schema"]
fwd_tbl = f"{catalog}.{schema}.acard_fwd_predictions"
spark.createDataFrame(FWD_PRED).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(fwd_tbl)

print(f"\nSaved {len(FWD_PRED):,} FWD predictions")
print(f"  CSV     : {fwd_csv}")
print(f"  Parquet : {fwd_parquet}")
print(f"  Table   : {fwd_tbl}")
print(f"  Columns : {keep_cols}")
print(f"\nPrediction summary (calibrated):")
for m in MODELS:
    col = f"prob_{m}_cal"
    print(f"  Model {m}: mean={FWD_PRED[col].mean():.4f}  std={FWD_PRED[col].std():.4f}  "
          f"min={FWD_PRED[col].min():.4f}  max={FWD_PRED[col].max():.4f}")

display(FWD_PRED.head(20))